<a href="https://colab.research.google.com/github/GiladBoudman/Haifa-Eco-Pulse/blob/main/ecohaifa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Imports & Setup

In [ ]:
import sys
import subprocess
from pathlib import Path
from scipy import stats as scipy_stats
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import matplotlib.ticker as ticker


try:
    import ee
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'earthengine-api'])
    import ee

try:
    # Authenticate and Initialize Google account & Project
    ee.Authenticate()
    PROJECT_ID = 'bionic-bond-172010'
    ee.Initialize(project=PROJECT_ID)
    print('Earth Engine initialized successfully.')
except Exception as e:
    print(f'Earth Engine initialization skipped or failed: {e}')

try:
    import geemap
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'geemap'])
    import geemap

try:
    from pykrige.ok import OrdinaryKriging
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pykrige'])
    from pykrige.ok import OrdinaryKriging

Earth Engine initialized successfully.


In [ ]:
#Region of Interest (ROI) - Define Haifa's centre coordinates and create a 5 km circular buffer around
haifa_lat = 32.7940
haifa_lon = 34.9896
haifa_point = ee.Geometry.Point([haifa_lon, haifa_lat])
haifa_roi = haifa_point.buffer(5000)

#GEE Data Pipelines

In [ ]:
#Sentinel-5P OFFL NO2 product
def get_no2_pipeline(roi, start_date, end_date):
    return (ee.ImageCollection('COPERNICUS/S5P/OFFL/L3_NO2')
            .filterBounds(roi)
            .filterDate(start_date, end_date)
            .map(lambda img: img.updateMask(img.select('cloud_fraction').lt(0.3)))
            .select('tropospheric_NO2_column_number_density'))
#Sentinel-2 SR Harmonised product
def get_ndvi_pipeline(roi, start_date, end_date):
    def mask_and_ndvi(image):
        qa = image.select('QA60')
        mask = qa.bitwiseAnd(1 << 10).eq(0).And(qa.bitwiseAnd(1 << 11).eq(0))
        ndvi = image.updateMask(mask).divide(10000).normalizedDifference(['B8', 'B4']).rename('NDVI')
        return image.addBands(ndvi).copyProperties(image, ['system:time_start'])
    return (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
            .filterBounds(roi)
            .filterDate(start_date, end_date)
            .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
            .map(mask_and_ndvi)
            .select('NDVI'))

#Monthly Data Fetching

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

#fetches mean NO2 and median NDVI for a single month
def fetch_month(roi, current):
    next_month = current + pd.DateOffset(months=1)
    start_str = current.strftime('%Y-%m-%d')
    end_str   = next_month.strftime('%Y-%m-%d')

    try:
        no2_col  = get_no2_pipeline(roi, start_str, end_str)
        ndvi_col = get_ndvi_pipeline(roi, start_str, end_str)

        # Skip months with no S2 imagery (causes empty image → select crash)
        if ndvi_col.size().getInfo() == 0:
            print(f"Skipping {start_str}: no S2 images")
            return None

        no2_image  = no2_col.mean().select('tropospheric_NO2_column_number_density').rename('NO2')
        ndvi_image = ndvi_col.median().select('NDVI')

        stats = no2_image.addBands(ndvi_image).reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=roi,
            scale=500,
            maxPixels=1e9
        ).getInfo() or {}

        no2_value  = stats.get('NO2')
        ndvi_value = stats.get('NDVI')
        if no2_value is None or ndvi_value is None:
            print(f"Skipping {start_str}: null stats (cloud-masked out)")
            return None

        return {'Month': current.strftime('%Y-%m'), 'NO2': float(no2_value), 'NDVI': float(ndvi_value)}

    except Exception as e:
        print(f"Error for {start_str}: {e}")
        return None

#fetches sampled spatial points (lon, lat, NO2) for one month – used later for Kriging interpolation.
def fetch_month_spatial(roi, current):
    next_month = current + pd.DateOffset(months=1)
    start_str = current.strftime('%Y-%m-%d')
    end_str   = next_month.strftime('%Y-%m-%d')

    try:
        no2_col = get_no2_pipeline(roi, start_str, end_str)
        if no2_col.size().getInfo() == 0:
            return None

        no2_image = no2_col.mean().rename('NO2')

        # Sample ~30 points within the ROI
        samples = no2_image.sample(
            region=roi,
            scale=1000,
            numPixels=30,
            geometries=True,
            seed=42
        )

        points = samples.getInfo()['features']
        if not points:
            return None

        rows = []
        for f in points:
            lon, lat = f['geometry']['coordinates']
            no2 = f['properties'].get('NO2')
            if no2 is not None:
                rows.append({'lon': lon, 'lat': lat, 'NO2': no2, 'Month': current.strftime('%Y-%m')})

        return rows if rows else None

    except Exception as e:
        print(f"Error for {start_str}: {e}")
        return None

#calls fetch_month for every month in a rolling window
def build_gee_monthly_table(roi, months_back=72):
    end   = pd.Timestamp.now().normalize()
    start = (end - pd.DateOffset(months=months_back)).replace(day=1)

    months = []
    current = start
    while current < end:
        months.append(current)
        current += pd.DateOffset(months=1)

    rows = []
    with ThreadPoolExecutor(max_workers=6) as executor:   # GEE allows ~6 concurrent
        futures = {executor.submit(fetch_month, roi, m): m for m in months}
        for future in as_completed(futures):
            result = future.result()
            if result:
                rows.append(result)

    return pd.DataFrame(rows).sort_values('Month').reset_index(drop=True)

#CSV Cache: Load or Fetch

In [ ]:
# To avoid re-querying GEE every run, results are cached in 'haifa_data.csv'.
#   - If the file exists and is valid (non-empty, correct columns), it is loaded.
#   - If the file is missing or corrupt, fresh data is fetched from GEE and saved.
CSV_PATH = 'haifa_data.csv'

def load_or_fetch(csv_path, roi):
    if Path(csv_path).exists():
        df = pd.read_csv(csv_path)
        # Validate: must have rows and required columns
        required_cols = {'Month', 'NDVI', 'NO2'}
        if not df.empty and required_cols.issubset(df.columns):
            print(f"Loaded {len(df)} rows from cache: {csv_path}")
            return df
        else:
            print(f"Cache invalid (shape={df.shape}, cols={list(df.columns)}), re-fetching...")
            Path(csv_path).unlink()  # Delete bad cache

    df = build_gee_monthly_table(roi)
    if df.empty:
        raise ValueError('No Earth Engine data was returned for the selected Haifa window.')
    df.to_csv(csv_path, index=False)
    print(f"Fetched {len(df)} rows from GEE and saved to {csv_path}")
    return df

df = load_or_fetch(CSV_PATH, haifa_roi)
df['Year'] = df['Month'].str.slice(0, 4)

Skipping 2020-11-01: no S2 images
Fetched 72 rows from GEE and saved to haifa_data.csv


In [ ]:
# Run this once to clear the bad cache, then re-run your notebook
# if Path(CSV_PATH).exists():
#     os.remove(CSV_PATH)
#     print("Cache cleared — will re-fetch from GEE")

#Output Containers

In [ ]:
# --- Output Containers ---
map_output   = widgets.Output()
graph_output = widgets.Output()
stats_output = widgets.Output()

In [ ]:
# --- Dropdowns & Selection State ---
available_years  = sorted(df['Year'].unique().tolist())
default_year     = available_years[-1]
default_months   = df[df['Year'] == default_year]['Month'].tolist()
default_month    = default_months[-1] if default_months else df['Month'].iloc[-1]

year_dropdown = widgets.Dropdown(
    options=available_years,
    value=default_year,
    description='Year',
    layout=widgets.Layout(width='180px', height='44px')
)
month_dropdown = widgets.Dropdown(
    options=default_months,
    value=default_month,
    description='Month',
    layout=widgets.Layout(width='220px', height='44px')
)

all_months = df['Month'].tolist()

SPECIAL_EVENTS = {
    'Custom (use slider)': None,
    'COVID-19 Peak (Mar–Apr 2020)': '2020-03',
    '7.10 War (Oct–Dec 2023)': '2023-10',
}

month_slider = widgets.SelectionRangeSlider(
    options=all_months,
    index=(0, len(all_months) - 1),
    description='Range',
    layout=widgets.Layout(width='500px')
)

selection_state = {'updating': False}
map_state = {'override_month': None}  # set when a special event is active
all_months = sorted(df['Month'].unique().tolist())

map_slider = widgets.IntSlider(
    min=0,
    max=len(all_months) - 1,
    value=all_months.index(default_month),
    description='Map month:',
    layout=widgets.Layout(width='450px'),
    style={'description_width': '90px'}
)

# Label to show the current month name next to the slider
map_slider_label = widgets.Label(value=f'📅 {default_month}')
map_slider_box = widgets.HBox([map_slider, map_slider_label])

apply_map_btn = widgets.Button(
    description='Apply',
    button_style='info',
    icon='check',
    layout=widgets.Layout(width='110px', height='36px')
)


In [ ]:
# triggered when the year dropdown changes.
#Updates month_dropdown options to match the new year,
def on_year_change(*_):
    selection_state['updating'] = True
    try:
        sync_month_options()
    finally:
        selection_state['updating'] = False
    sync_slider_to_dropdowns()
    refresh_views()

#triggered when the month dropdown changes.
def on_month_change(*_):
    if selection_state['updating']:
        return
    sync_slider_to_dropdowns()
    refresh_views()

In [ ]:
#repopulates month_dropdown when the year changes
def sync_month_options(*_):
    selected_year = year_dropdown.value
    months_for_year = df[df['Year'] == selected_year]['Month'].tolist()
    current_value = month_dropdown.value
    month_dropdown.options = months_for_year
    if current_value in months_for_year:
        month_dropdown.value = current_value
    elif months_for_year:
        month_dropdown.value = months_for_year[-1]
def on_slider_change(*_):
    selected = all_months[map_slider.value]
    map_slider_label.value = f'📅 {selected}'

#updates the map month label as the slider moves.
def sync_slider_to_dropdowns():
    target = month_dropdown.value
    if target in all_months:
        idx = all_months.index(target)
        if map_slider.value != idx:
            map_slider.value = idx
            map_slider_label.value = f'📅 {target}'

#called when "Apply" is clicked – clears any special
#event override and syncs the slider value back to
#the year/month dropdowns, then refreshes.
def on_apply_map(_):
    event_dropdown.value = 'Custom (use slider)'
    map_state['override_month'] = None   # ← clear override
    selected = all_months[map_slider.value]
    year = selected[:4]
    selection_state['updating'] = True
    try:
        if year in year_dropdown.options:
            year_dropdown.value = year
        sync_month_options()
        if selected in month_dropdown.options:
            month_dropdown.value = selected
    finally:
        selection_state['updating'] = False
    refresh_views()

apply_map_btn.on_click(on_apply_map)

event_dropdown = widgets.Dropdown(
    options=list(SPECIAL_EVENTS.keys()),
    value='Custom (use slider)',
    description='Special events:',
    layout=widgets.Layout(width='320px'),
    style={'description_width': '110px'}
)

#handle special event selection, enabling or disabling the slider/apply button accordingly
def on_event_change(*_):
    selected_event = event_dropdown.value
    month = SPECIAL_EVENTS[selected_event]
    if month is None:
        map_state['override_month'] = None
        map_slider.disabled = False
        apply_map_btn.disabled = False
        refresh_views()
        return

    map_slider.disabled = True
    apply_map_btn.disabled = True
    map_state['override_month'] = month  # ← store it regardless of CSV range

    year = month[:4]
    selection_state['updating'] = True
    try:
        if year in year_dropdown.options:
            year_dropdown.value = year
        sync_month_options()
        if month in month_dropdown.options:
            month_dropdown.value = month
        if month in all_months:
            map_slider.value = all_months.index(month)
            map_slider_label.value = f'📅 {month}'
        else:
            map_slider_label.value = f'📅 {month} (event)'  # outside CSV range
    finally:
        selection_state['updating'] = False
    refresh_views()

def on_event_dropdown_change(*_):
    if event_dropdown.value == 'Custom (use slider)':
        map_slider.disabled = False
        apply_map_btn.disabled = False
    else:
        on_event_change()

#Observer registrations at the bottom wire all the above callbacks to their
# respective widget 'value' change events.
event_dropdown.observe(on_event_dropdown_change, names='value')
event_dropdown.observe(on_event_dropdown_change, names='value')
year_dropdown.observe(on_year_change, names='value')
month_dropdown.observe(on_month_change, names='value')
map_slider.observe(on_slider_change, names='value')

In [ ]:
#plot_kriging_map: given a list of sampled {lon, lat, NO2} points for one
# month, fits an Ordinary Kriging model (spherical variogram) and produces a
# contourf heatmap over a 60×60 interpolation grid.
def plot_kriging_map(spatial_rows, selected_month):
    """Given a list of {lon, lat, NO2} dicts, krig and plot."""
    df_sp = pd.DataFrame(spatial_rows)

    lons = df_sp['lon'].values
    lats = df_sp['lat'].values
    vals = df_sp['NO2'].values

    # Build interpolation grid
    # grid_lon = np.linspace(lons.min(), lons.max(), 60)
    # grid_lat = np.linspace(lats.min(), lats.max(), 60)
    grid_lon = np.linspace(34.9896 - 0.045, 34.9896 + 0.045, 60)
    grid_lat = np.linspace(32.7940 - 0.045, 32.7940 + 0.045, 60)

    ok = OrdinaryKriging(
        lons, lats, vals,
        variogram_model='spherical',
        verbose=False, enable_plotting=False
    )
    z, ss = ok.execute('grid', grid_lon, grid_lat)

    fig, ax = plt.subplots(figsize=(8, 7))
    c = ax.contourf(grid_lon, grid_lat, z, levels=20, cmap='RdYlGn_r')
    ax.scatter(lons, lats, c='black', s=20, zorder=5, label='Sample points')
    plt.colorbar(c, ax=ax, label='NO₂ (mol/m²)')
    ax.set_title(f'Kriging Interpolation — NO₂ over Haifa ({selected_month})', fontweight='bold')
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.legend()
    fig.tight_layout()
    return fig

In [ ]:
# ── helpers ──────────────────────────────────────────────────────────────────
# converts a 'YYYY-MM' string to (start_date, end_date)
def get_date_range(selected_month: str) -> tuple[str, str]:
    month_start = selected_month + '-01'
    month_end = (pd.Timestamp(month_start) + pd.DateOffset(months=1)).strftime('%Y-%m-%d')
    return month_start, month_end

# maps a month number to a season name
def get_season(month_str: str) -> str:
    m = int(month_str.split('-')[1])
    if m in [12, 1, 2]:  return 'Winter'
    if m in [3, 4, 5]:   return 'Spring'
    if m in [6, 7, 8]:   return 'Summer'
    return 'Autumn'

# computes a dynamic NO2 colour-scale range
def get_no2_viz_params_from_df(selected_month: str) -> dict:
    # Use surrounding months for a stable range (±2 months)
    idx = all_months.index(selected_month) if selected_month in all_months else None
    if idx is not None:
        window = all_months[max(0, idx-2) : idx+3]
        subset = df[df['Month'].isin(window)]['NO2']
    else:
        subset = df['NO2']  # fallback: full dataset range

    return {
        'min': float(subset.min()),
        'max': float(subset.max()),
        'palette': ['#313695', '#4575b4', '#74add1', '#fdae61', '#f46d43', '#d73027', '#a50026'],
    }

# static visualisation parameters for the NDVI layer
NDVI_VIZ = {
    'min': 0.0, 'max': 1.0, 'opacity': 0.7,
    'palette': ['#a52a2a', '#d4a96a', '#ffffcc', '#78c679', '#006837'],
}


In [ ]:
# ── map ───────────────────────────────────────────────────────────────────────
#constructs a geemap.Map centred on Haifa and adds two layers
def build_map(roi, month_start: str, month_end: str, selected_month: str):
    no2_image  = get_no2_pipeline(roi, month_start, month_end).mean()
    ndvi_image = get_ndvi_pipeline(roi, month_start, month_end).median()
    no2_viz    = get_no2_viz_params_from_df(selected_month)  # ← from CSV, no GEE round-trip

    m = geemap.Map(
        center=[haifa_lat, haifa_lon], zoom=36,
        zoom_control=False, scroll_wheel_zoom=False,
        dragging=False, double_click_zoom=False,
        touch_zoom=False, keyboard=False,
    )
    m.addLayer(ndvi_image.clip(roi), NDVI_VIZ,  f'NDVI {selected_month}', True, 0.85)
    m.addLayer(no2_image.clip(roi),  no2_viz,   f'NO2 {selected_month}',  True, 0.7)
    m.centerObject(roi, 12)
    m.add_colorbar(vis_params=no2_viz,  label='NO2 (mol/m²)', orientation='horizontal', position='bottomleft')
    m.add_colorbar(vis_params=NDVI_VIZ, label='NDVI',         orientation='horizontal', position='bottomright')
    return m

In [ ]:
# ── figures ───────────────────────────────────────────────────────────────────
def plot_timeseries_and_scatter(year_df: pd.DataFrame, selected_year: int) -> plt.Figure:
    r, p_value = scipy_stats.pearsonr(year_df['NO2'], year_df['NDVI'])

    fig, (ax1, ax_scatter) = plt.subplots(1, 2, figsize=(16, 5))
    fig.patch.set_facecolor('#F8FAFC')
    fig.suptitle(f'Haifa Air Quality & Vegetation Analysis — {selected_year}',
                 fontsize=14, fontweight='bold', y=1.02)

    # ── Time series ──
    ax1.set_title('Monthly NO2 vs NDVI Trend', fontsize=12, fontweight='bold', pad=10)
    ax1.plot(year_df['Month'], year_df['NO2'], color='#DC2626', marker='o', linewidth=2, label='NO2')
    ax1.set_ylabel('NO2 (mol/m²)', color='#991B1B', fontsize=10)
    ax1.set_xlabel('Month', fontsize=10)
    ax1.tick_params(axis='y', colors='#991B1B')
    ax1.tick_params(axis='x', rotation=45, labelsize=8)
    ax1.grid(True, alpha=0.2)

    ax2 = ax1.twinx()
    ax2.plot(year_df['Month'], year_df['NDVI'], color='#15803D', marker='s', linewidth=2, label='NDVI')
    ax2.set_ylabel('NDVI', color='#166534', fontsize=10)
    ax2.tick_params(axis='y', colors='#166534')

    # Combined legend for both axes
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper right', fontsize=9)

    # ── Scatter + regression ──
    ax_scatter.set_title('NO2 / NDVI Correlation', fontsize=12, fontweight='bold', pad=10)
    m, b = np.polyfit(year_df['NO2'], year_df['NDVI'], 1)
    x_line = np.linspace(year_df['NO2'].min(), year_df['NO2'].max(), 100)
    ax_scatter.scatter(year_df['NO2'], year_df['NDVI'], color='#0F766E', zorder=5, label='Monthly data')
    ax_scatter.plot(x_line, m * x_line + b, color='#DC2626', linewidth=2, linestyle='--', label='Regression line')
    ax_scatter.set_xlabel('NO2 (mol/m²)', fontsize=10)
    ax_scatter.set_ylabel('NDVI', fontsize=10)
    ax_scatter.set_title(f'NO2 / NDVI Correlation\nPearson r = {r:.3f}  (p = {p_value:.3f})',
                         fontsize=12, fontweight='bold', pad=10)
    ax_scatter.grid(True, alpha=0.2)
    ax_scatter.legend(fontsize=9)

    significance = 'Significant (p < 0.05)' if p_value < 0.05 else 'Not significant (p ≥ 0.05)'
    ax_scatter.annotate(
        significance, xy=(0.05, 0.05), xycoords='axes fraction',
        fontsize=9, color='#64748B',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='#F1F5F9', edgecolor='#CBD5E1'),
    )

    fig.tight_layout()
    return fig


def plot_anova_bars(df: pd.DataFrame,selected_year: int = None) -> plt.Figure:
    df = df.copy()
    df['Season'] = df['Month'].apply(get_season)

    season_order = ['Winter', 'Spring', 'Summer', 'Autumn']
    groups  = [df[df['Season'] == s]['NO2'].dropna().values for s in season_order]
    f_stat, anova_p = scipy_stats.f_oneway(*groups)

    means  = [g.mean() for g in groups]
    stds   = [g.std()  for g in groups]
    colors = ['#93C5FD', '#86EFAC', '#FCD34D', '#F97316']

    fig, ax = plt.subplots(figsize=(8, 5))
    fig.patch.set_facecolor('#F8FAFC')
    ax.set_facecolor('#F8FAFC')

    significance = 'Significant (p < 0.05)' if anova_p < 0.05 else 'Not significant (p ≥ 0.05)'
    fig.suptitle('Seasonal NO2 Distribution — All Years', fontsize=13, fontweight='bold')
    title_suffix = f' — {selected_year}' if selected_year else ''
    ax.set_title(f'ANOVA {title_suffix}: F = {f_stat:.2f}, p = {anova_p:.3f}  |  {significance}',
                 fontsize=10, color='#475569', pad=8)

    bars = ax.bar(season_order, means, yerr=stds, capsize=5,
                  color=colors, edgecolor='white', linewidth=1.2,
                  error_kw={'linewidth': 1.5})

    for bar, mean in zip(bars, means):
        ax.text(
            bar.get_x() + bar.get_width() / 2,   # ← fixed typo: was get_wisdth()
            bar.get_height() + max(stds) * 0.05,
            f'{mean:.2e}', ha='center', va='bottom', fontsize=9, color='#334155',
        )

    ax.set_ylabel('Mean NO2 (mol/m²)', fontsize=11, fontweight='bold', color='#334155')
    ax.set_xlabel('Season', fontsize=11, fontweight='bold')
    ax.grid(True, axis='y', alpha=0.2)

    fig.tight_layout()
    return fig

In [ ]:
# ── stats table ───────────────────────────────────────────────────────────────

def build_stats_table(df: pd.DataFrame, year_df: pd.DataFrame,
                      selected_year: int, selected_month: str):

    current_row = df[(df['Year'] == selected_year) & (df['Month'] == selected_month)].iloc[0]

    # ── Pearson ──────────────────────────────────────────────────────────────
    r, p_pearson = scipy_stats.pearsonr(year_df['NO2'], year_df['NDVI'])
    pearson_sig  = 'Yes ✓' if p_pearson < 0.05 else 'No ✗'

    # ── ANOVA ─────────────────────────────────────────────────────────────────
    year_df = year_df.copy()
    year_df['Season'] = year_df['Month'].apply(get_season)
    season_order = ['Winter', 'Spring', 'Summer', 'Autumn']
    groups = [year_df[year_df['Season'] == s]['NO2'].dropna().values for s in season_order]
    # Need at least 2 seasons with data to run ANOVA
    valid_groups = [g for g in groups if len(g) > 0]
    if len(valid_groups) >= 2:
        f_stat, p_anova = scipy_stats.f_oneway(*valid_groups)
        anova_sig = 'Yes ✓' if p_anova < 0.05 else 'No ✗'
    else:
        f_stat, p_anova, anova_sig = None, None, 'Not enough data'

    # ── Selected month values ─────────────────────────────────────────────────
    basic_rows = [
        ('Selected period',  '',                        ''),
        ('Year',             selected_year,             ''),
        ('Month',            selected_month,            ''),
        ('NO₂ this month',   f"{current_row['NO2']:.4e}", 'mol/m²'),
        ('NDVI this month',  f"{current_row['NDVI']:.4f}", '0–1 index'),

        ('Annual summary',   '',                        ''),
        ('Months of data',   len(year_df),              ''),
        ('NO₂ mean',         f"{year_df['NO2'].mean():.4e}", 'mol/m²'),
        ('NO₂ min',          f"{year_df['NO2'].min():.4e}", 'mol/m²'),
        ('NO₂ max',          f"{year_df['NO2'].max():.4e}", 'mol/m²'),
        ('NDVI mean',        f"{year_df['NDVI'].mean():.4f}", '0–1 index'),
        ('NDVI min',         f"{year_df['NDVI'].min():.4f}", '0–1 index'),
        ('NDVI max',         f"{year_df['NDVI'].max():.4f}", '0–1 index'),

        ('Pearson correlation', '',                     ''),
        ('r value',          f"{r:.4f}",                '-1 to 1  (0 = no link)'),
        ('p value',          f"{p_pearson:.4f}",        'p < 0.05 = significant'),
        ('Significant?',     pearson_sig,               ''),
        ('Interpretation',   'Strong' if abs(r) > 0.6 else 'Moderate' if abs(r) > 0.3 else 'Weak',
                             'negative = pollution ↑ vegetation ↓'),

        ('ANOVA (seasonal NO₂)', '',                   ''),
        ('F statistic',      f"{f_stat:.4f}" if f_stat is not None else '—', 'higher = more seasonal difference'),
        ('p value',          f"{p_anova:.4f}" if p_anova is not None else '—', 'p < 0.05 = significant'),
        ('Significant?',     anova_sig,                 ''),
        ('Interpretation',   'Seasons differ significantly in NO₂' if (p_anova is not None and p_anova < 0.05)
                             else 'No significant seasonal pattern', ''),
    ]

    rows_out = []
    for label, value, note in basic_rows:
        rows_out.append({'Metric': label, 'Value': value, 'Note': note})

    stats_df = pd.DataFrame(rows_out)

    # Style: section headers (Value == '') get a teal background
    def style_rows(row):
        if row['Value'] == '':
            return ['background-color: #CCFBF1; color: #0F766E; font-weight: 700; font-size: 13px;'] * 3
        return [''] * 3

    return (stats_df.style
            .hide(axis='index')
            .apply(style_rows, axis=1)
            .set_properties(**{'font-size': '13px', 'padding': '6px 10px'})
            .set_table_styles([
                {'selector': 'th', 'props': [
                    ('background-color', '#0F766E'), ('color', 'white'),
                    ('font-size', '13px'), ('padding', '8px 10px')
                ]},
                {'selector': 'td', 'props': [('border-bottom', '1px solid #E2E8F0')]},
            ]))

In [ ]:
def refresh_views(*_):
    selected_year  = year_dropdown.value
    selected_month = month_dropdown.value
    year_df = df[df['Year'] == selected_year].copy()

    # Use event override if active, otherwise use slider
    map_month = map_state['override_month'] or all_months[map_slider.value]
    map_start, map_end = get_date_range(map_month)

    with map_output:
        clear_output(wait=True)
        display(build_map(haifa_roi, map_start, map_end, map_month))

    with graph_output:
        clear_output(wait=True)
        fig1 = plot_timeseries_and_scatter(year_df, selected_year)
        display(fig1)
        plt.close(fig1)
        fig2 = plot_anova_bars(year_df, selected_year)
        display(fig2)
        plt.close(fig2)

    with stats_output:
        clear_output(wait=True)

        # ── compute values first ──────────────────────────────────────────
        no2_val  = float(df[(df['Year'] == selected_year) & (df['Month'] == selected_month)]['NO2'].iloc[0])
        ndvi_val = float(df[(df['Year'] == selected_year) & (df['Month'] == selected_month)]['NDVI'].iloc[0])
        r_val, _ = scipy_stats.pearsonr(year_df['NO2'], year_df['NDVI'])

        no2_display = no2_val * 1e5
        display(HTML(f"""
        <div style='margin-bottom: 14px; padding: 16px 20px; border-radius: 14px;
                    background: linear-gradient(135deg, #0F766E 0%, #0EA5E9 100%);
                    box-shadow: 0 4px 14px rgba(14, 165, 233, 0.3);'>
          <div style='font-size: 20px; font-weight: 800; color: #FFFFFF; letter-spacing: 0.01em;'>
            📊 Statistics — {selected_month}
          </div>
          <div style='font-size: 13px; color: rgba(255,255,255,0.85); margin-top: 5px;'>
            Pearson and ANOVA computed on <strong>{selected_year}</strong> monthly data ({len(year_df)} months).
            Scroll down for full breakdown.
          </div>
        </div>
        """))

        display(HTML(f"""
        <div style='display: flex; gap: 16px; align-items: flex-start;'>

          <!-- ── Left: scale legend ─────────────────────────────────────────── -->
          <div style='min-width: 200px; max-width: 220px; display: flex; flex-direction: column; gap: 12px;'>

            <!-- NO2 scale -->
            <div style='padding: 14px; border-radius: 14px; background: #F8FAFC; border: 1px solid #E2E8F0;'>
              <div style='font-size: 11px; font-weight: 700; letter-spacing: 0.1em;
                          text-transform: uppercase; color: #64748B; margin-bottom: 10px;'>NO₂ Scale</div>
              {''.join([
                f"""<div style='display:flex; align-items:center; gap:8px; margin-bottom:6px;'>
                      <div style='width:10px; height:10px; border-radius:50%; background:{c}; flex-shrink:0;'></div>
                      <div style='font-size:12px; color:#334155;'>{label}</div>
                      {"<div style='margin-left:auto; font-size:11px; font-weight:700; color:" + col + ";'> ◀ now</div>" if active else ""}
                    </div>"""
                    for c, label, col, active in [
                    ('#15803D', '< 4.0 — Clean',      '#15803D', no2_display < 4.0),
                    ('#84CC16', '4.0–5.5 — Normal',   '#65A30D', 4.0 <= no2_display < 5.5),
                    ('#F97316', '5.5–6.5 — Elevated', '#C2410C', 5.5 <= no2_display < 6.5),
                    ('#B91C1C', '> 6.5 — High',       '#B91C1C', no2_display >= 6.5),
                ]
              ])}
            </div>

            <!-- NDVI scale -->
            <div style='padding: 14px; border-radius: 14px; background: #F8FAFC; border: 1px solid #E2E8F0;'>
              <div style='font-size: 11px; font-weight: 700; letter-spacing: 0.1em;
                          text-transform: uppercase; color: #64748B; margin-bottom: 10px;'>NDVI Scale</div>
              {''.join([
                f"""<div style='display:flex; align-items:center; gap:8px; margin-bottom:6px;'>
                      <div style='width:10px; height:10px; border-radius:50%; background:{c}; flex-shrink:0;'></div>
                      <div style='font-size:12px; color:#334155;'>{label}</div>
                      {"<div style='margin-left:auto; font-size:11px; font-weight:700; color:" + col + ";'> ◀ now</div>" if active else ""}
                    </div>"""
                for c, label, col, active in [
                  ('#B91C1C', '< 0.15 — Bare',          '#B91C1C', ndvi_val < 0.15),
                  ('#F97316', '0.15–0.25 — Sparse',     '#C2410C', 0.15 <= ndvi_val < 0.25),
                  ('#84CC16', '0.25–0.40 — Moderate',   '#65A30D', 0.25 <= ndvi_val < 0.40),
                  ('#15803D', '> 0.40 — Healthy',        '#15803D', ndvi_val >= 0.40),
                ]
              ])}
            </div>

            <!-- Correlation scale -->
            <div style='padding: 14px; border-radius: 14px; background: #F8FAFC; border: 1px solid #E2E8F0;'>
              <div style='font-size: 11px; font-weight: 700; letter-spacing: 0.1em;
                          text-transform: uppercase; color: #64748B; margin-bottom: 10px;'>Pearson r Scale</div>
              {''.join([
                f"""<div style='display:flex; align-items:center; gap:8px; margin-bottom:6px;'>
                      <div style='width:10px; height:10px; border-radius:50%; background:{c}; flex-shrink:0;'></div>
                      <div style='font-size:12px; color:#334155;'>{label}</div>
                      {"<div style='margin-left:auto; font-size:11px; font-weight:700; color:" + col + ";'> ◀ now</div>" if active else ""}
                    </div>"""
                for c, label, col, active in [
                  ('#0F766E', '|r| > 0.6 — Strong',     '#0F766E', abs(r_val) >= 0.6),
                  ('#84CC16', '0.3–0.6 — Moderate',     '#65A30D', 0.3 <= abs(r_val) < 0.6),
                  ('#94A3B8', '< 0.3 — Weak',           '#64748B', abs(r_val) < 0.3),
                ]
              ])}
              <div style='margin-top: 8px; padding-top: 8px; border-top: 1px solid #E2E8F0;
                          font-size: 11px; color: #64748B; line-height: 1.5;'>
                {'Negative — pollution ↑, vegetation ↓' if r_val < 0 else 'Positive — both rise together'}
              </div>
            </div>

          </div>

          <!-- ── Right: table (injected by Python below) ────────────────────── -->
          <div id='stats-table-target' style='flex: 1; min-width: 0;'></div>

        </div>
        """))

        display(build_stats_table(df, year_df, selected_year, selected_month))

In [ ]:
def go_kriging(_):
    open_dashboard(kriging_panel)
    selected_month = month_dropdown.value
    with kriging_output:
        clear_output(wait=True)
        display(HTML("<div style='color:#0F766E; font-weight:700;'>⏳ Fetching spatial samples...</div>"))
        rows = fetch_month_spatial(haifa_roi, pd.Timestamp(selected_month + '-01'))
        clear_output(wait=True)
        if not rows:
            display(HTML("<div style='color:red;'>No spatial data returned for this month.</div>"))
            return
        fig = plot_kriging_map(rows, selected_month)
        display(fig)
        plt.close(fig)

In [ ]:
# ── colour tokens (reused across home + all panels) ──────────────────────────
BG_PAGE    = '#F0F4F8'   # soft blue-grey page background
BG_CARD    = '#FFFFFF'
TEAL_DARK  = '#0F766E'
TEAL_MID   = '#14B8A6'
NAVY       = '#0F172A'
SLATE      = '#334155'
SLATE_LIGHT= '#64748B'
BORDER     = '#CBD5E1'

# Apply background to the whole notebook output area
display(HTML("""
<style>
  .jp-Cell-outputArea, .output_area, div.output {
      background: #F0F4F8 !important;
  }
</style>
"""))

home_title = widgets.HTML(f"""
<div style='
    padding: 28px 32px;
    border-radius: 24px;
    background: linear-gradient(135deg, #0F172A 0%, #134E4A 60%, #0F766E 100%);
    color: white;
    box-shadow: 0 20px 48px rgba(15, 23, 42, 0.22);
    width: 100%;
    box-sizing: border-box;
'>
  <div style='font-size: 11px; letter-spacing: 0.16em; text-transform: uppercase; opacity: 0.65; margin-bottom: 6px;'>
    Braude College · Environmental Data Science
  </div>
  <div style='font-size: 32px; font-weight: 800; line-height: 1.1; margin-bottom: 14px;'>
    Haifa Air Quality Explorer
  </div>
  <div style='font-size: 15px; line-height: 1.75; opacity: 0.88; max-width: 720px; margin-bottom: 22px;'>
    This dashboard tracks two environmental indicators over Haifa using satellite data from
    <strong>Google Earth Engine</strong>:<br>
    &bull; <strong>NO₂</strong> — nitrogen dioxide concentration in the troposphere, a marker of air pollution
    from traffic, industry, and combustion.<br>
    &bull; <strong>NDVI</strong> — Normalized Difference Vegetation Index, a measure of how green and healthy
    the vegetation is. Higher NDVI = healthier plants.<br><br>
    The goal is to explore whether periods of high air pollution correspond to lower vegetation health,
    and how that relationship changes across seasons and significant events.
  </div>
  <div style='display: grid; grid-template-columns: repeat(4, minmax(0, 1fr)); gap: 14px;'>
    <div style='background: rgba(255,255,255,0.1); border: 1px solid rgba(255,255,255,0.2); border-radius: 16px; padding: 16px;'>
      <div style='font-size: 20px; margin-bottom: 4px;'>🗺️ Map</div>
      <div style='font-size: 13px; font-weight: 700; margin-bottom: 6px;'>Satellite View</div>
      <div style='font-size: 12px; opacity: 0.85; line-height: 1.6;'>
        See NO₂ and NDVI layers rendered over Haifa for any month. Use the slider or pick a special event to compare periods.
      </div>
    </div>
    <div style='background: rgba(255,255,255,0.1); border: 1px solid rgba(255,255,255,0.2); border-radius: 16px; padding: 16px;'>
      <div style='font-size: 20px; margin-bottom: 4px;'>📈 Trend</div>
      <div style='font-size: 13px; font-weight: 700; margin-bottom: 6px;'>Monthly Analysis</div>
      <div style='font-size: 12px; opacity: 0.85; line-height: 1.6;'>
        Track how NO₂ and NDVI changed month by month for a selected year, and see their Pearson correlation in a scatter plot.
      </div>
    </div>
    <div style='background: rgba(255,255,255,0.1); border: 1px solid rgba(255,255,255,0.2); border-radius: 16px; padding: 16px;'>
      <div style='font-size: 20px; margin-bottom: 4px;'>📊 Stats</div>
      <div style='font-size: 13px; font-weight: 700; margin-bottom: 6px;'>Summary Numbers</div>
      <div style='font-size: 12px; opacity: 0.85; line-height: 1.6;'>
        Review mean, min, max and correlation values for any year and month. Includes seasonal ANOVA breakdown.
      </div>
    </div>
    <div style='background: rgba(255,255,255,0.1); border: 1px solid rgba(255,255,255,0.2); border-radius: 16px; padding: 16px;'>
      <div style='font-size: 20px; margin-bottom: 4px;'>🌐 Kriging</div>
      <div style='font-size: 13px; font-weight: 700; margin-bottom: 6px;'>Spatial Interpolation</div>
      <div style='font-size: 12px; opacity: 0.85; line-height: 1.6;'>
        View a continuous NO₂ heatmap over Haifa for the selected month, interpolated from satellite sample points using Ordinary Kriging.
      </div>
    </div>
  </div>
</div>
""")

how_to = widgets.HTML(f"""
<div style='
    padding: 16px 20px;
    border-radius: 16px;
    background: {BG_CARD};
    border: 1px solid {BORDER};
    color: {SLATE};
    font-size: 15px;
    line-height: 1.7;
    width: 100%;
    box-sizing: border-box;
'>
  <div style='font-weight: 700; font-size: 16px; color: {TEAL_DARK}; margin-bottom: 8px;'>🧭 How to use</div>
  <ol style='margin: 0; padding-left: 20px;'>
    <li><strong>Pick a year and month</strong> using the dropdowns below — this sets the time window for the graphs and stats.</li>
    <li><strong>Open the Map</strong> to see NO₂ and NDVI satellite layers over Haifa. Use the month slider inside the map tab to explore different months visually without affecting the graphs.</li>
    <li><strong>Open the Trend graph</strong> to see how pollution and vegetation moved together across the selected year, including a Pearson correlation scatter plot.</li>
    <li><strong>Open Statistics</strong> for a numerical summary — mean, min, max, Pearson correlation, and a seasonal ANOVA breakdown of NO₂.</li>
    <li><strong>Open Kriging Interpolation</strong> to see a spatial heatmap of NO₂ over Haifa for the selected month, generated by fitting a spherical variogram to satellite sample points.</li>
  </ol>
""")

time_picker_header = widgets.HTML(f"""
<div style='font-size: 12px; font-weight: 700; letter-spacing: 0.1em; text-transform: uppercase;
            color: {TEAL_DARK}; margin-bottom: 10px; text-align: center;'>
  Choose a time window
</div>
""")

time_picker_row = widgets.HBox(
    [year_dropdown, month_dropdown],
    layout=widgets.Layout(gap='12px', justify_content='center', flex_flow='row wrap',
                          align_items='center', width='100%')
)
time_picker_card = widgets.VBox(
    [time_picker_header, time_picker_row],
    layout=widgets.Layout(padding='16px 18px', border=f'1px solid {BORDER}',
                          border_radius='18px', background_color=BG_CARD,
                          box_shadow='0 8px 20px rgba(15,23,42,0.06)',
                          align_items='center', width='100%')
)

kriging_output = widgets.Output()
launch_kriging = widgets.Button(
    description='Open Kriging', button_style='danger', icon='globe',
    layout=widgets.Layout(width='200px', height='52px')
)
kriging_info = widgets.HTML(f"""
<div style='
    padding: 16px 20px;
    border-radius: 16px;
    background: {BG_CARD};
    border: 1px solid {BORDER};
    color: {SLATE};
    font-size: 15px;
    line-height: 1.7;
    width: 100%;
    box-sizing: border-box;
    margin-bottom: 12px;
'>
  <div style='font-weight: 700; font-size: 16px; color: {TEAL_DARK}; margin-bottom: 8px;'>🌐 What you're seeing</div>
  <p style='margin: 0 0 8px 0;'>
    This heatmap shows <strong>estimated NO₂ concentration</strong> across Haifa for the selected month,
    produced by <strong>Ordinary Kriging</strong> — a geostatistical interpolation method.
  </p>
  <ul style='margin: 0; padding-left: 20px;'>
    <li><strong>Black dots</strong> are the actual satellite sample points (~30 per month) fetched from Sentinel-5P via Google Earth Engine.</li>
    <li><strong>The colored surface</strong> is Kriging's continuous prediction between those points, using a spherical variogram to model how NO₂ correlation decays with distance.</li>
    <li><strong>Red areas</strong> indicate higher NO₂ concentration; <strong>green areas</strong> indicate lower concentration.</li>
    <li>The interpolation grid is <strong>60×60</strong>, spanning the bounding box of the sample points.</li>
  </ul>
</div>
""")

# ── Map button — standalone, clearly separated ────────────────────────────────
map_note = widgets.HTML("""
<div style='
    font-size: 11px; color: #64748B; text-align: center;
    margin-top: 4px; font-style: italic;
'>
  Has its own month slider inside
</div>
""")

launch_map = widgets.Button(
    description='Open Map', button_style='info', icon='map-marker',
    layout=widgets.Layout(width='200px', height='52px')
)
#############

#############
map_block = widgets.VBox(
    [launch_map, map_note],
    layout=widgets.Layout(align_items='center', gap='2px', width='100%')
)

# ── Divider ───────────────────────────────────────────────────────────────────
divider = widgets.HTML("""
<div style='
    display: flex; align_items: center; gap: 10px;
    color: #94A3B8; font-size: 12px; text-align: center;
    padding: 0 8px;
'>
  <div style='flex:1; height:1px; background:#CBD5E1;'></div>
  <div style='white-space: nowrap;'>dropdowns control these ↓</div>
  <div style='flex:1; height:1px; background:#CBD5E1;'></div>
</div>
""")

# ── Graphs + Stats — grouped under the dropdown ───────────────────────────────
launch_graph = widgets.Button(
    description='Open Trend', button_style='success', icon='line-chart',
    layout=widgets.Layout(width='200px', height='52px')
)
launch_stats = widgets.Button(
    description='Open Statistics', button_style='warning', icon='bar-chart',
    layout=widgets.Layout(width='200px', height='52px')
)

controlled_row = widgets.HBox(
    [launch_graph, launch_stats,launch_kriging],
    layout=widgets.Layout(gap='14px', justify_content='center',
                          flex_flow='row wrap', align_items='center')
)

controlled_block = widgets.VBox(
    [time_picker_card, controlled_row],
    layout=widgets.Layout(
        gap='14px', align_items='center', padding='16px',
        border='2px dashed #14B8A6', border_radius='18px',
        width='100%', box_sizing='border-box'
    )
)

launch_row = widgets.VBox(
    [
        map_block,
        divider,
        controlled_block,
    ],
    layout=widgets.Layout(
        gap='16px', align_items='center',
        width='100%'
    )
)

home_page = widgets.VBox([
    home_title,
    how_to,
    launch_row,
], layout=widgets.Layout(gap='16px', padding='8px 0 20px 0', align_items='center', width='100%'))

In [ ]:
back_home = widgets.Button(description='Back to Home', button_style='warning', icon='home', layout=widgets.Layout(width='180px', height='44px'))
top_bar = widgets.HBox([back_home], layout=widgets.Layout(justify_content='flex-start', margin='0 0 14px 0'))

#map_view = widgets.VBox([map_output])
graph_view = widgets.VBox([graph_output])
stats_view = widgets.VBox([stats_output])

shell_style = widgets.Layout(
    padding='20px',
    border=f'1px solid {BORDER}',
    border_radius='24px',
    background_color=BG_PAGE,          # ← was #FFFFFF
    box_shadow='0 18px 44px rgba(15, 23, 42, 0.10)',
    width='100%'
)
panel_style = widgets.Layout(
    padding='20px',
    border=f'1px solid {BORDER}',
    border_radius='20px',
    background_color=BG_CARD,
    box_shadow='0 10px 24px rgba(15, 23, 42, 0.06)',
    width='100%'
)



kriging_panel = widgets.VBox([kriging_info, kriging_output], layout=panel_style)
launch_kriging.on_click(go_kriging)

main_view = widgets.VBox(
    [home_page],
    layout=widgets.Layout(padding='16px', background_color=BG_PAGE, width='100%')
)
dashboard_shell = widgets.VBox([main_view], layout=shell_style)
map_panel = widgets.VBox([
    widgets.HTML("<div style='font-size:13px; font-weight:700; color:#0F766E; margin-bottom:8px;'>Drag to change map month</div>"),
    event_dropdown,
    widgets.HBox([map_slider_box, apply_map_btn], layout=widgets.Layout(align_items='center', gap='12px')),
    map_output,
], layout=panel_style)
graph_panel = widgets.VBox([graph_output], layout=panel_style)
stats_panel = widgets.VBox([stats_output], layout=panel_style)

def open_home(_=None):
    main_view.children = [home_page]

def open_dashboard(view):
    if isinstance(view, int):
        view = [map_panel, graph_panel, stats_panel][view]
    main_view.children = [top_bar, view]

def go_map(_):
    open_dashboard(map_panel)

def go_graph(_):
    open_dashboard(graph_view)

def go_stats(_):
    open_dashboard(stats_view)

def go_home(_):
    open_home()

launch_map.on_click(go_map)
launch_graph.on_click(go_graph)
launch_stats.on_click(go_stats)
back_home.on_click(go_home)

display(dashboard_shell)
refresh_views()